In [ ]:
"""###Import Libraries"""
from tensorflow.keras.layers import (Conv2D, Add,Activation,
                                     Dropout,Dense,Flatten,Input,BatchNormalization,
                                      ReLU,MaxPooling2D,Concatenate,GlobalAveragePooling2D
                                     )
from tensorflow.keras.models import Sequential
from tensorflow.keras import optimizers, losses, metrics, regularizers, callbacks
from tensorflow.keras.models import Model
import tensorflow as tf
import numpy as np
import os
import sys
from pathlib import Path
import h5py; 

util_dir = os.getcwd()+ "/utils"
sys.path.append(util_dir)
import Utility
import importlib
importlib.reload(Utility)

print(sys.version)
print("TF version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("h5py:", h5py.__version__)

data_path = os.getcwd()+ "/data/"
model_path =  os.getcwd()+ "/models/"

In [ ]:
model_name = 'ECG_Echo_CNN'
print(model_name)

In [ ]:
"""#### Import Data"""

calssificatin_type = {"binary":1,"superclasses":5,"subclasses":23}
classification_name="binary"
no_of_classes=calssificatin_type[classification_name]

x_train = np.load(data_path + 'x_tr_prep.npy',allow_pickle=True)
x_val = np.load(data_path + 'x_val_prep.npy',allow_pickle=True)
x_test  = np.load(data_path + 'x_tst_prep.npy',allow_pickle=True)

y_train_raw = np.load(data_path + 'y_tr.npy',allow_pickle=True)
y_val_raw = np.load(data_path + 'y_val.npy',allow_pickle=True)
y_test_raw  = np.load(data_path + 'y_tst.npy',allow_pickle=True)

# Composite
y_train = y_train_raw[:,0]
y_test = y_test_raw[:,0]
y_val = y_val_raw[:,0]


ntrain, no_of_leads, ntime, nch = x_train.shape

print("x_train :", x_train.shape)
print("y_train :", y_train.shape)
print("x_val :", x_val.shape)
print("y_val :", y_val.shape)
print("x_test  :", x_test.shape)
print("y_test  :", y_test.shape)
print('Data loaded')


In [ ]:
"""#### Model"""
tf.keras.backend.clear_session()

ntrain, no_of_leads, ntime, nch = x_train.shape
input= Input(shape=(no_of_leads, ntime, 1))

conv1 = Conv2D(filters=32, kernel_size=(1,7),strides=(1, 1))(input)
batch1 = BatchNormalization()(conv1)
relu1 = ReLU()(batch1)

conv2 = Conv2D(filters=64, kernel_size=(1,5),strides=(1, 1))(relu1)
batch2 = BatchNormalization()(conv2)
relu2 = ReLU()(batch2)
drop2 = Dropout(rate=0.1)(relu2)
conv2 = Conv2D(filters=64, kernel_size=(1,5),strides=(1, 2))(drop2)

max1 = MaxPooling2D(pool_size=(1,9),strides=(1,2))(relu1)
conv_ = Conv2D(64, (1, 1))(max1)
conc1 = Add()([conv2, conv_])

batch3 = BatchNormalization()(conc1)
relu3 = ReLU()(batch3)
drop3 = Dropout(rate=0.1)(relu3)
conv3 = Conv2D(filters=64, kernel_size=(1,5),strides=(1, 1))(drop3)
batch3 = BatchNormalization()(conv3)
relu3 = ReLU()(batch3)
drop3 = Dropout(rate=0.1)(relu3)
conv3 = Conv2D(filters=64, kernel_size=(1,5),strides=(1, 2))(drop3)

max2 = MaxPooling2D(pool_size=(1,9),strides=(1,2))(conc1)
conc2 = Add()([conv3, max2])

batch3 = BatchNormalization()(conc2)
relu3 = ReLU()(batch3)
drop3 = Dropout(rate=0.1)(relu3)
conv3 = Conv2D(filters=128, kernel_size=(1,5),strides=(1, 1))(drop3)
batch3 = BatchNormalization()(conv3)
relu3 = ReLU()(batch3)
drop3 = Dropout(rate=0.1)(relu3)
conv3 = Conv2D(filters=128, kernel_size=(1,5),strides=(1, 2))(drop3)

max3 = MaxPooling2D(pool_size=(1,9),strides=(1,2))(conc2)
conv_ = Conv2D(128, (1, 1))(max3)
conc3 = Add()([conv3, conv_])

batch3 = BatchNormalization()(conc3)
relu3 = ReLU()(batch3)
drop3 = Dropout(rate=0.1)(relu3)
conv3 = Conv2D(filters=128, kernel_size=(1,5),strides=(1, 1))(drop3)
batch3 = BatchNormalization()(conv3)
relu3 = ReLU()(batch3)
drop3 = Dropout(rate=0.1)(relu3)
conv3 = Conv2D(filters=128, kernel_size=(1,5),strides=(1, 2))(drop3)

max4 = MaxPooling2D(pool_size=(1,9),strides=(1,2))(conc3)
conc4 = Add()([conv3, max4])
          
conv3 = Conv2D(filters=128, kernel_size=(no_of_leads, 1))(conc4)
X = BatchNormalization()(conv3)
X = ReLU()(X)
X = GlobalAveragePooling2D()(X)

X = Flatten()(X)

X = Dense(units=128, kernel_regularizer=tf.keras.regularizers.L2(0.005))(X)
X = BatchNormalization()(X)
X = ReLU()(X)
X = Dropout(rate=0.1)(X)

X = Dense(units=64, kernel_regularizer=tf.keras.regularizers.L2(0.009))(X)
X = BatchNormalization()(X)
X = ReLU()(X)
X = Dropout(rate=0.15)(X)

output = Dense(no_of_classes, activation='sigmoid')(X)
model = Model(inputs=input, outputs=output)

model.summary()


In [ ]:
"""#### Train Model"""
import tensorflow as tf 
early    = callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
reducelr = callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3)

callback = [early, reducelr]
model.compile(optimizer = optimizers.Adam(learning_rate=0.0005),
              loss = losses.BinaryCrossentropy(),
              metrics = [metrics.BinaryAccuracy(), metrics.AUC(curve='ROC', multi_label=True)])

history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=100, batch_size=32, callbacks=callback)

In [ ]:
y_pred_train = model.predict(x_train)
y_pred_test  = model.predict(x_test)

In [ ]:
Utility.save_model_package(
    model=model,
    model_name=model_name,
    x_test=x_test,
    y_test=y_test,
    x_train=x_train,
    y_train=y_train,
    y_pred_train=y_pred_train,
    y_pred_test=y_pred_test,
    history=history,
    base_dir = model_path
 )


In [ ]:
y_pred_val = model.predict(x_val)
np.save(model_path+model_name+'/y_pred_val.npy',np.array(y_pred_val))
np.save(model_path+model_name+'/y_val.npy',y_val)